# Teste Interativo da Classe FlightSeasonality

Notebook de validação da classe `FlightSeasonality` (análise de padrões sazonais de atrasos).

Análises testadas:
1. Sazonalidade mensal (taxa, delay médio/mediana, p90, volume)
2. Padrão semanal (dias da semana)
3. Padrão por período do dia
4. Heatmaps cruzados (mês × dia, mês × período, dia × período)
5. Sazonalidade por companhia aérea
6. Sazonalidade dos aeroportos anômalos (cruzamento com notebook 06)
7. Combinações críticas (pior cenário)

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto para permitir importar a classe do módulo notebooks
sys.path.append(str(Path(".").resolve().parent))

from notebooks.sazonalidade import FlightSeasonality

In [ ]:
# Inicializa o analisador.
# - top_n_airlines=8 → analisa as 8 companhias com mais voos
# - min_voos_critical=100 → mínimo de voos por combinação no ranking de cenários críticos
# - anomalous_airports pode ser definido depois via set_anomalous_airports([...])
season = FlightSeasonality(
    input_path="../data/processed/flights_model.parquet",
    top_n_airlines=8,
    min_voos_critical=100,
)

In [ ]:
# 1. Carregar dados e adicionar flags de atraso
df = season.load_data()

## Sazonalidade mensal

In [ ]:
monthly = season.compute_monthly()
monthly

In [ ]:
season.plot_monthly()

## Padrão semanal

In [ ]:
weekly = season.compute_weekly()
weekly

In [ ]:
season.plot_weekly()

## Padrão por período do dia

In [ ]:
period = season.compute_period()
period

In [ ]:
season.plot_period()

## Heatmaps cruzados

In [ ]:
# Heatmap 1: Dia da Semana × Mês
season.plot_cross_month_dow()

In [ ]:
# Heatmap 2: Período do Dia × Mês
season.plot_cross_month_period()

In [ ]:
# Heatmap 3: Período do Dia × Dia da Semana
season.plot_cross_dow_period()

## Sazonalidade por companhia aérea

In [ ]:
# Calcula a tabela companhia × mês
season.compute_airline_monthly()
print("Top companhias selecionadas:", season.top_airlines)

In [ ]:
# Heatmap: companhia × mês
season.plot_airline_heatmap()

In [ ]:
# Linha temporal por companhia
season.plot_airline_lines()

## Sazonalidade dos aeroportos anômalos

Cruzamento com o notebook 06: o comportamento atípico é constante ou concentrado em certos meses?

Use `set_anomalous_airports([...])` para passar a lista vinda do notebook 06.

In [ ]:
# Exemplo: aeroportos anômalos detectados pelo notebook 06
# Substitua pela lista real obtida do detector.anomalos_consenso.index.tolist()
season.set_anomalous_airports([
    "PSG", "YAK", "CDV", "BTM", "ASE", "PSE", "EGE", "BPT", "LWS",
])

In [ ]:
season.plot_anomalous_seasonality()

## Combinações críticas (pior cenário)

In [ ]:
# Top 15 combinações (mês + dia + período) com piores taxas
critical = season.compute_critical_combinations(top_n=15)
critical[["combinacao", "taxa_atraso", "media_delay", "qtd_voos"]].head(10)

In [ ]:
season.plot_critical_combinations(top_n=15)

## Alternativa: pipeline completo com `run_all`

In [ ]:
# Roda tudo de uma vez (mas você precisa setar anomalous_airports antes se quiser a seção 6)
# season_full = FlightSeasonality(
#     input_path="../data/processed/flights_model.parquet",
#     anomalous_airports=["PSG", "YAK", "CDV", "BTM", "ASE"],
# )
# season_full.run_all(show_plots=True)